# Road Damage Detection Final Notebook

This notebook documents the full workflow for building a road damage detection pipeline with YOLO models: dataset preparation, training, evaluation, visual diagnostics, and export.

The project compares five models:
- YOLOv8n
- YOLOv8s
- YOLOv8m
- YOLOv5s-u
- YOLOv11n

The output includes the exported weights, evaluation summary, a damage-density heatmap, and a downloadable model package.

In [ ]:
# Install project dependencies
!pip install ultralytics seaborn -q

import os
import cv2
import json
import shutil
import random
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from ultralytics import YOLO

print("Environment ready. Using PyTorch and Ultralytics.")

In [ ]:
# Configuration paths
DATASET_SOURCE = "/kaggle/input/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes/data/images"
LABELS_SOURCE = "/kaggle/input/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes/data/labels-YOLO"
DEST_BASE = "/kaggle/working/road-damage-split"
EXPORT_DIR = "/kaggle/working/exported_models"
RUNS_DIR = "/kaggle/working/road-damage-runs"

os.makedirs(EXPORT_DIR, exist_ok=True)

# Clear and recreate the dataset split
if os.path.exists(DEST_BASE):
    shutil.rmtree(DEST_BASE)

for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(DEST_BASE, "images", split), exist_ok=True)
    os.makedirs(os.path.join(DEST_BASE, "labels", split), exist_ok=True)

# Prepare train/val/test split
image_files = [f for f in os.listdir(DATASET_SOURCE) if f.endswith((".jpg", ".png", ".jpeg"))]
random.seed(42)
random.shuffle(image_files)

train_idx = int(len(image_files) * 0.70)
val_idx = int(len(image_files) * 0.85)

splits = {
    "train": image_files[:train_idx],
    "val": image_files[train_idx:val_idx],
    "test": image_files[val_idx:],
}

for split, files in splits.items():
    for img_file in files:
        source_img = os.path.join(DATASET_SOURCE, img_file)
        target_img = os.path.join(DEST_BASE, "images", split, img_file)
        shutil.copy(source_img, target_img)

        label_file = img_file.rsplit(".", 1)[0] + ".txt"
        source_label = os.path.join(LABELS_SOURCE, label_file)
        target_label = os.path.join(DEST_BASE, "labels", split, label_file)
        if os.path.exists(source_label):
            shutil.copy(source_label, target_label)

print(f"Dataset split complete:")
print(f"Train: {len(splits['train'])} images")
print(f"Val: {len(splits['val'])} images")
print(f"Test: {len(splits['test'])} images")

# Build the YOLO data config
config = {
    "path": DEST_BASE,
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 3,
    "names": {0: "Pothole", 1: "Crack", 2: "Manhole"},
}

with open(os.path.join(DEST_BASE, "data.yaml"), "w") as file:
    yaml.safe_dump(config, file, sort_keys=False)

print(f"YAML config saved to {os.path.join(DEST_BASE, 'data.yaml')}")

In [ ]:
# Quick dataset inspection
print(f"Scanning dataset root: {DATASET_SOURCE}")
for root, dirs, files in os.walk(DATASET_SOURCE):
    level = root.replace(DATASET_SOURCE, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}📁 {os.path.basename(root)}/")
    if level < 3:
        for d in sorted(dirs):
            print(f"{indent}  📂 {d}")

In [ ]:
# Risk logic for severity interpretation
def assess_risk(cls_name, area_percent):
    if cls_name == "Pothole":
        if area_percent > 5.0:
            return "Critical"
        elif area_percent > 1.0:
            return "High"
        return "Medium"
    elif cls_name == "Crack":
        if area_percent > 10.0:
            return "High"
        elif area_percent > 3.0:
            return "Medium"
        return "Low"
    elif cls_name == "Manhole":
        return "Low"
    return "Unknown"

print("Risk assessment function ready.")

In [ ]:
# Train multiple YOLO backbones
models_to_train = {
    "yolov8n": "yolov8n.pt",
    "yolov8s": "yolov8s.pt",
    "yolov8m": "yolov8m.pt",
    "yolov5su": "yolov5su.pt",
    "yolov11n": "yolo11n.pt",
}

common_args = {
    "data": os.path.join(DEST_BASE, "data.yaml"),
    "epochs": 80,
    "imgsz": 640,
    "batch": 16,
    "device": 0,
    "patience": 15,
    "optimizer": "AdamW",
    "lr0": 0.001,
    "lrf": 0.01,
    "warmup_epochs": 3,
    "mosaic": 1.0,
    "mixup": 0.1,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "flipud": 0.0,
    "fliplr": 0.5,
    "scale": 0.5,
    "project": RUNS_DIR,
    "exist_ok": True,
}

for name, weights in models_to_train.items():
    print(f"\n{'=' * 42}\nTraining {name}\n{'=' * 42}")
    model = YOLO(weights)
    common_args["name"] = name
    model.train(**common_args)

    best_pt = os.path.join(RUNS_DIR, name, "weights", "best.pt")
    if os.path.exists(best_pt):
        shutil.copy(best_pt, os.path.join(EXPORT_DIR, f"{name}_best.pt"))
        print(f"Saved best checkpoint for {name}")

print("\nAll models trained and exported.")

In [ ]:
# Evaluate all trained models on the test set
results_summary = []

for name in models_to_train.keys():
    model_path = os.path.join(EXPORT_DIR, f"{name}_best.pt")
    if not os.path.exists(model_path):
        print(f"Skipping {name}: weights not found")
        continue

    model = YOLO(model_path)
    metrics = model.val(split="test", verbose=False)

    map50 = float(round(metrics.box.map50, 4))
    map5095 = float(round(metrics.box.map, 4))
    precision = float(round(metrics.box.mp, 4))
    recall = float(round(metrics.box.mr, 4))

    avg_area_list = []
    risk_counts = {"Low": 0, "Medium": 0, "High": 0, "Critical": 0}
    test_images = os.listdir(os.path.join(DEST_BASE, "images", "test"))

    for img_name in test_images[: min(100, len(test_images))]:
        img_path = os.path.join(DEST_BASE, "images", "test", img_name)
        result = model.predict(img_path, verbose=False)[0]
        h, w = result.orig_shape

        for box in result.boxes:
            cls_id = int(box.cls[0])
            cls_name = model.names[cls_id]
            xyxy = box.xyxy[0].cpu().numpy()
            area_percent = float(((xyxy[2] - xyxy[0]) * (xyxy[3] - xyxy[1])) / (w * h) * 100)
            avg_area_list.append(area_percent)

            risk = assess_risk(cls_name, area_percent)
            if risk in risk_counts:
                risk_counts[risk] += 1

    avg_area = float(round(np.mean(avg_area_list), 2)) if avg_area_list else 0.0

    results_summary.append(
        {
            "model": name,
            "mAP50": map50,
            "mAP5095": map5095,
            "precision": precision,
            "recall": recall,
            "avg_damage_area_percent": avg_area,
            "risk_distribution": risk_counts,
        }
    )

summary_path = os.path.join(EXPORT_DIR, "results_summary.json")
with open(summary_path, "w") as output_file:
    json.dump(results_summary, output_file, indent=4)

summary_df = pd.DataFrame(results_summary)
print("\nFinal evaluation summary:")
print(summary_df[["model", "mAP50", "mAP5095", "precision", "recall", "avg_damage_area_percent"]])

In [ ]:
# Plot the model comparison
summary_df = pd.DataFrame(json.load(open(os.path.join(EXPORT_DIR, "results_summary.json"))))

plt.figure(figsize=(10, 5))
plt.bar(summary_df["model"], summary_df["mAP50"], color="steelblue")
plt.title("mAP50 Comparison by Model")
plt.ylabel("mAP50")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# Generate a spatial heatmap of detected damage hotspots
best_model = "yolov8s_best.pt"
best_model_path = os.path.join(EXPORT_DIR, best_model)
test_dir = os.path.join(DEST_BASE, "images", "test")
heatmap_path = os.path.join(EXPORT_DIR, "damage_heatmap.png")

model = YOLO(best_model_path)
all_test_images = os.listdir(test_dir)
sample_img = cv2.imread(os.path.join(test_dir, all_test_images[0]))
h, w = sample_img.shape[:2]
accumulator = np.zeros((h, w), dtype=np.float32)

for img_name in all_test_images[: min(200, len(all_test_images))]:
    img_path = os.path.join(test_dir, img_name)
    result = model.predict(img_path, verbose=False)[0]
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
        cls_id = int(box.cls[0])
        if cls_id in [0, 1]:
            accumulator[y1:y2, x1:x2] += 1

accumulator_norm = cv2.normalize(accumulator, None, 0, 255, cv2.NORM_MINMAX)
heatmap = cv2.applyColorMap(accumulator_norm.astype(np.uint8), cv2.COLORMAP_JET)
overlay = cv2.addWeighted(sample_img, 0.65, heatmap, 0.35, 0)
cv2.imwrite(heatmap_path, overlay)

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.title("Damage Density Heatmap")
plt.axis("off")
plt.show()

In [ ]:
# Visualize a few prediction examples against ground truth
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

model = YOLO(best_model_path)
random_test_images = random.sample(os.listdir(test_dir), 3)

for i, img_name in enumerate(random_test_images):
    img_path = os.path.join(test_dir, img_name)
    gt_img = cv2.imread(img_path)
    label_path = os.path.join(DEST_BASE, "labels", "test", img_name.rsplit(".", 1)[0] + ".txt")

    if os.path.exists(label_path):
        h, w = gt_img.shape[:2]
        with open(label_path, "r") as label_file:
            for line in label_file:
                cls, xc, yc, bw, bh = map(float, line.strip().split())
                x1 = int((xc - bw / 2) * w)
                y1 = int((yc - bh / 2) * h)
                x2 = int((xc + bw / 2) * w)
                y2 = int((yc + bh / 2) * h)
                color = (0, 255, 0) if cls == 0 else (0, 0, 255) if cls == 1 else (255, 0, 0)
                cv2.rectangle(gt_img, (x1, y1), (x2, y2), color, 2)
                cv2.putText(gt_img, f"GT: {model.names[int(cls)]}", (x1, max(0, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    axes[i].imshow(cv2.cvtColor(gt_img, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"Ground Truth: {img_name}")
    axes[i].axis("off")

    prediction = model.predict(img_path, verbose=False)[0]
    predicted_img = prediction.plot()
    axes[i + 3].imshow(cv2.cvtColor(predicted_img, cv2.COLOR_BGR2RGB))
    axes[i + 3].set_title(f"Prediction: {img_name}")
    axes[i + 3].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Package the exported model folder for easy sharing
zip_path = "/kaggle/working/road_damage_package.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)

!cd /kaggle/working && zip -r road_damage_package.zip exported_models/

print("\nPackaging complete. Downloadable model package created.")
print(f"Archive: {zip_path}")

## Final notes

This pipeline is ready for repeatable experiments and deployment. The exported weights in the `exported_models` folder can be used directly by the Streamlit app for inference, while the JSON summary file enables rapid comparison across models and training runs.